[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C14_DL_Theory_Data_Course/03_data_pipeline/03_data_pipeline.ipynb)

# 03 · 数据流水线（用 numpy/stdlib 从零搭）

目标：从零实现并量化 **清洗 → 质量过滤 → MinHash 近重复去重 → shuffle 缓冲 → 分片 → 端到端流水线**, 用 `assert` 钉死每步效果。

路线：清洗 → 质量过滤启发式 → shingle+Jaccard → MinHash(估计收敛到真 Jaccard) → shuffle 缓冲偏差 → 端到端流水线+吞吐 → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊(tiny-shakespeare 去重/质量)。

> 心智模型：**每一步 = 读文档流→处理/过滤→出文档流**；MinHash 的魔法 = **签名相等率无偏估计 Jaccard**, 把 O(N²) 去重降到近线性。

## 1 · 清洗：去 HTML、控制字符、规范空白

最朴素却最值钱的一步。脏数据(HTML 残骸、控制字符、乱七八糟的空白)会污染下游一切。

In [ ]:
import re, hashlib
import numpy as np
rng = np.random.default_rng(0)

def clean_text(text):
    '''去 HTML 标签、控制字符, 规范空白。'''
    text = re.sub(r'<[^>]+>', ' ', text)              # 去 HTML 标签
    text = re.sub(r'[\x00-\x1f\x7f]', ' ', text)    # 去控制字符
    text = re.sub(r'\s+', ' ', text).strip()          # 规范空白
    return text

dirty = '<p>Hello   world!</p>\n\t<a href=x>link</a>\x00 extra'
clean = clean_text(dirty)
print('脏:', repr(dirty))
print('净:', repr(clean))
assert '<' not in clean and '>' not in clean, 'HTML 标签应被去掉'
assert '\x00' not in clean and '\t' not in clean, '控制字符应被去掉'
assert '  ' not in clean, '多余空白应被压成单空格'
assert clean == 'Hello world! link extra'
print('✅ 清洗: HTML/控制字符/多余空白全部规范化')

## 2 · 质量过滤：Gopher 风格启发式

用一组启发式剔除低质文档。核心信号: **长度、符号比、重复行比、停用词比**。
每条都是弱特征, 任一触发即过滤。没有完美过滤器——阈值是召回与精确的权衡。

In [ ]:
STOPWORDS = set('the a an of to in is are and or but for with on at by it this that'.split())

def quality_features(text):
    words = text.split()
    n = max(len(words), 1)
    symbols = sum(c in '#@$%^&*<>{}[]|\\/~`' for c in text)
    lines = [ln for ln in text.split('\n') if ln.strip()]
    dup_line_ratio = 1.0 - len(set(lines)) / max(len(lines), 1)
    stop_ratio = sum(w.lower() in STOPWORDS for w in words) / n
    return dict(n_words=len(words), symbol_ratio=symbols / max(len(text), 1),
                dup_line_ratio=dup_line_ratio, stop_ratio=stop_ratio)

def passes_quality(text, min_words=10, max_symbol=0.10, max_dup_line=0.3, min_stop=0.05):
    f = quality_features(text)
    return (f['n_words'] >= min_words and f['symbol_ratio'] <= max_symbol
            and f['dup_line_ratio'] <= max_dup_line and f['stop_ratio'] >= min_stop)

good = 'the cat sat on the mat and the dog ran in the park with a ball today'
spam = '###BUY NOW### @@@ $$$ click here !!! cheap deals %%% <<<>>>'
short = 'too short'
print('优质文本通过?', passes_quality(good))
print('SEO垃圾通过? ', passes_quality(spam))
print('过短文本通过?', passes_quality(short))
assert passes_quality(good), '自然语言应通过'
assert not passes_quality(spam), '符号堆砌的垃圾应被过滤'
assert not passes_quality(short), '过短文档应被过滤'
print('✅ 质量过滤: 自然语言留下, 符号垃圾/过短被剔除')

## 3 · shingle + Jaccard：量化两文档有多像

把文档表示成 **k-词 shingle 集合**, 用 **Jaccard = |A∩B|/|A∪B|** 度量重叠。
用 shingle(而非词袋)是为了捕捉**局部词序**: 词相同但顺序乱的文档 shingle 差异大。

In [ ]:
def shingles(text, k=3):
    '''所有长度 k 的连续词组(集合)。'''
    toks = text.split()
    if len(toks) < k:
        return {tuple(toks)}
    return {tuple(toks[i:i + k]) for i in range(len(toks) - k + 1)}

def jaccard(a, b):
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)

d1 = 'the quick brown fox jumps over the lazy dog and runs away'
d2 = 'the quick brown fox leaps over the lazy dog and runs away'   # 近重复(改1词)
d3 = 'machine learning data pipelines deduplication and quality filtering matter'  # 无关
A, B, C = shingles(d1), shingles(d2), shingles(d3)
print(f'Jaccard(d1,d2 近重复) = {jaccard(A, B):.3f}')
print(f'Jaccard(d1,d3 无关)   = {jaccard(A, C):.3f}')
print(f'Jaccard(d1,d1 自己)   = {jaccard(A, A):.3f}')
assert jaccard(A, A) == 1.0, '自己和自己 Jaccard=1'
assert jaccard(A, B) > 0.5, '近重复 Jaccard 高'
assert jaccard(A, C) < 0.05, '无关文档 Jaccard≈0'
# 词序敏感: 同词不同序, shingle Jaccard 低
shuffled = 'fox brown quick the over jumps dog lazy the away runs and'
assert jaccard(A, shingles(shuffled)) < jaccard(A, B), '打乱词序后 shingle 相似度下降'
print('✅ Jaccard+shingle: 近重复高、无关低、对词序敏感')

## 4 · MinHash：用签名无偏估计 Jaccard

**核心定理**: `P(min_A h == min_B h) = Jaccard(A,B)`。用 m 个独立哈希得签名, **两签名相等位的比例 = Jaccard 的无偏估计**(方差~J(1-J)/m)。

哈希族: `h(x) = (a*x + b) mod P`(P 大素数), shingle 先用 md5 转成整数。这把「比大集合」压成「比短向量」。

In [ ]:
PRIME = (1 << 61) - 1                      # 大素数

def shingle_to_int(sh):
    return int(hashlib.md5(repr(sorted(sh)).encode()).hexdigest(), 16)

def make_hash_family(m, seed=0):
    r = np.random.default_rng(seed)
    a = r.integers(1, PRIME, size=m, dtype=np.int64)
    b = r.integers(0, PRIME, size=m, dtype=np.int64)
    return a, b

def minhash_signature(shingle_set, a, b):
    '''返回长度 m 的签名: 每个哈希下所有 shingle 的最小哈希值。'''
    if not shingle_set:
        return np.full(len(a), PRIME, dtype=np.int64)
    xs = np.array([shingle_to_int(s) % PRIME for s in shingle_set], dtype=np.int64)
    sig = np.empty(len(a), dtype=np.int64)
    for i in range(len(a)):
        sig[i] = int(((a[i] * xs + b[i]) % PRIME).min())
    return sig

def estimate_jaccard(sig1, sig2):
    return float(np.mean(sig1 == sig2))

true_J = jaccard(A, B)
print(f'真 Jaccard(d1,d2) = {true_J:.3f}')
print(f"{'哈希数 m':>10}{'MinHash估计':>14}{'|误差|':>10}")
for m in [50, 200, 800]:
    a, b = make_hash_family(m, seed=0)
    est = estimate_jaccard(minhash_signature(A, a, b), minhash_signature(B, a, b))
    print(f'{m:>10}{est:>14.3f}{abs(est - true_J):>10.3f}')

a, b = make_hash_family(800, seed=0)
est_800 = estimate_jaccard(minhash_signature(A, a, b), minhash_signature(B, a, b))
est_unrelated = estimate_jaccard(minhash_signature(A, a, b), minhash_signature(C, a, b))
assert abs(est_800 - true_J) < 0.1, 'm=800 时估计应接近真 Jaccard'
assert est_unrelated < 0.1, '无关文档 MinHash 估计≈0'
print('✅ MinHash: 签名相等率无偏估计 Jaccard, m 越大越准 —— 大集合压成短向量')

## 5 · 近重复去重：用 MinHash 把重复塌缩成一份

有了 MinHash 估计, 近重复去重 = 对文档两两估计 Jaccard, 超阈值则判为重复、只留一份。
(真实流水线用 LSH 分桶避免 O(N²); 这里玩具规模直接两两比, 但用的是 MinHash 估计而非精确 Jaccard。)

In [ ]:
def dedup_near(docs, m=256, threshold=0.4, seed=0):
    '''返回保留的文档下标(去掉与已保留文档近重复的)。'''
    a, b = make_hash_family(m, seed)
    sigs = [minhash_signature(shingles(d), a, b) for d in docs]
    keep = []
    for i in range(len(docs)):
        is_dup = any(estimate_jaccard(sigs[i], sigs[j]) >= threshold for j in keep)
        if not is_dup:
            keep.append(i)
    return keep

docs = [
    'the quick brown fox jumps over the lazy dog every single morning in the green park near the river',
    'the quick brown fox leaps over the lazy dog every single morning in the green park near the river',  # 近重复 0(改1词)
    'the quick brown fox jumps over the lazy dog every single morning in the green park near the river too',  # 近重复 0(加1词)
    'machine learning models need clean and carefully deduplicated training data to generalize well in practice',
    'machine learning models require clean and carefully deduplicated training data to generalize well in practice',  # 近重复 3
    'a completely unrelated sentence about cooking delicious pasta with fresh tomato sauce and basil leaves',
]
kept = dedup_near(docs, m=256, threshold=0.4)
print(f'原始 {len(docs)} 篇 -> 去重后 {len(kept)} 篇, 保留下标 {kept}')
assert len(kept) == 3, '应塌缩成 3 个去重簇(2组近重复+1独立)'
assert 0 in kept and 3 in kept and 5 in kept, '每簇保留第一篇'
assert 1 not in kept and 2 not in kept and 4 not in kept, '近重复被去掉'
print('✅ 近重复去重: 6 篇塌缩成 3 个去重簇, 精确去重(哈希)做不到这点')

## 6 · shuffle 缓冲：用有限内存换近似全局随机

流式数据无法全局 shuffle。**shuffle 缓冲**: 维护 B 条的缓冲区, 每次随机取1条、补1条。
**缓冲越大越接近真随机**。用「位移」量化: 把有序流过缓冲, 测每条平均移动多远。B=1 不打乱、B=N 等于全局 shuffle。

In [ ]:
def shuffle_buffer(stream, buf_size, seed=0):
    rng = np.random.default_rng(seed)
    it = iter(stream)
    buf = []
    for _ in range(buf_size):
        try: buf.append(next(it))
        except StopIteration: break
    out = []
    for item in it:
        j = int(rng.integers(len(buf)))
        out.append(buf[j]); buf[j] = item        # 取一条, 补一条
    rng.shuffle(buf); out.extend(buf)            # 流尽, 清空缓冲
    return out

def mean_displacement(out, N):
    '''有序流(0..N-1)经 shuffle 后, 每个值离原位置的平均距离。'''
    pos = np.empty(N)
    for i, v in enumerate(out):
        pos[v] = i
    return float(np.mean(np.abs(pos - np.arange(N))))

N = 1000
stream = list(range(N))                          # 故意有序(最坏情况)
print(f"{'缓冲 B':>8}{'平均位移':>12}{'随机程度':>20}")
disps = {}
for B in [1, 10, 100, 1000]:
    d = mean_displacement(shuffle_buffer(stream, B, seed=0), N)
    disps[B] = d
    print(f'{B:>8}{d:>12.1f}{"  " + ("几乎不动" if B==1 else "接近全局" if B>=N else "局部打乱"):>20}')
assert disps[1] == 0.0, 'B=1: 完全不打乱(位移0)'
assert disps[10] < disps[100] < disps[1000], '缓冲越大位移越大(越随机)'
# B=N 应接近全局 shuffle 的理论位移(~N/3)
assert disps[1000] > N / 5, 'B=N 接近全局 shuffle'
print('✅ shuffle 缓冲: B=1 不打乱、B 越大越随机、B=N≈全局 shuffle —— 内存换随机性的权衡')

---
## ✏️ 练习 1：近重复去重

实现 `count_unique_clusters(docs, m, threshold)`：用 MinHash 估计, 返回去重后剩余的文档**簇数**。
复用 `dedup_near` 的逻辑(或自己写), 返回 `len(kept)`。

In [ ]:
def count_unique_clusters(docs, m=200, threshold=0.7, seed=0):
    # TODO: 用 MinHash 签名, 贪心保留: 与已保留文档估计 Jaccard 都 < threshold 才保留
    #       返回保留的文档数
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
docs_t = [
    'data deduplication makes language models better and safer to deploy',
    'data deduplication makes language models better and safer to use',   # 近重复
    'an entirely different topic about photosynthesis in green plants',
]
k = count_unique_clusters(docs_t, m=200, threshold=0.6)
print(f'去重后簇数 = {k}')
assert k == 2, '3 篇(2近重复+1独立) -> 2 簇'
# 阈值很高时, 轻微改动不算重复 -> 全保留
assert count_unique_clusters(docs_t, m=200, threshold=0.99) == 3
print('✅ 练习 1 通过: 近重复去重簇数随阈值变化')

## ✏️ 练习 2：shuffle 缓冲偏差

实现 `block_mixing(n_blocks, block_size, buf_size)`：把 `n_blocks` 个「主题块」(每块 `block_size` 条同标签)拼成有序流, 过 shuffle 缓冲后, 返回**相邻两条来自不同块的比例**(混合度, 越高越打散)。

In [ ]:
def block_mixing(n_blocks, block_size, buf_size, seed=0):
    # TODO: stream = [块0 重复block_size次, 块1 ..., ...](标签 0,0,..,1,1,..)
    #       过 shuffle_buffer(stream, buf_size), 算相邻元素标签不同的比例
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
mix_small = block_mixing(5, 100, buf_size=1)      # 不打乱
mix_big   = block_mixing(5, 100, buf_size=500)    # 充分打乱
print(f'缓冲=1   相邻异块比例 = {mix_small:.3f}')
print(f'缓冲=500 相邻异块比例 = {mix_big:.3f}')
assert mix_small < 0.05, '不打乱时主题扎堆, 几乎没有相邻异块'
assert mix_big > 0.5, '充分打乱后相邻多来自不同块'
assert mix_big > mix_small, '缓冲越大混合度越高'
print('✅ 练习 2 通过: 小缓冲打不散主题块 -> 训练有偏(这是真实陷阱)')

## ✏️ 练习 3：质量过滤

实现 `filter_corpus(docs)`：对一批文档应用清洗 + 质量过滤, 返回 `(保留的文档列表, 过滤统计 dict)`。
统计 dict 含 `kept`、`removed`、`removal_rate`。

In [ ]:
def filter_corpus(docs):
    # TODO: 先 clean_text 每篇, 再 passes_quality 过滤
    #       返回 (kept_docs, {'kept':.., 'removed':.., 'removal_rate':..})
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
raw = [
    '<p>the cat sat on the mat and played with a ball in the garden today</p>',  # 好(清洗后)
    '###SPAM### $$$ @@@ !!! buy now cheap %%%',                                   # 垃圾
    'tiny',                                                                       # 过短
    'the dog ran fast across the field and jumped over the fence with joy now',   # 好
]
kept, stats = filter_corpus(raw)
print('保留', stats['kept'], '篇, 过滤', stats['removed'], '篇, 过滤率', f"{stats['removal_rate']:.0%}")
assert stats['kept'] == 2 and stats['removed'] == 2
assert abs(stats['removal_rate'] - 0.5) < 1e-9
assert all('<' not in d for d in kept), '保留的文档应已清洗'
print('✅ 练习 3 通过: 清洗+过滤并报告统计')

## ✏️ 练习 4：吞吐统计

实现 `throughput(items, process_fn)`：对每个 item 调用 `process_fn`, 返回 `(处理条数, 每秒处理条数)`。
用它对比一个「快」和一个「慢」处理函数的吞吐, 定位瓶颈。

In [ ]:
import time
def throughput(items, process_fn):
    # TODO: 计时处理所有 items, 返回 (len(items), len(items)/耗时秒)
    #       提示: t0=time.perf_counter(); 处理; dt=time.perf_counter()-t0
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
items = ['doc ' * 20] * 2000
n_fast, tps_fast = throughput(items, lambda x: len(x))            # 快
n_slow, tps_slow = throughput(items, lambda x: clean_text(x * 5)) # 慢(正则)
print(f'快处理: {tps_fast:.0f} 条/秒')
print(f'慢处理: {tps_slow:.0f} 条/秒')
assert n_fast == n_slow == 2000
assert tps_fast > tps_slow, '简单操作吞吐应高于正则清洗'
assert tps_fast > 0 and tps_slow > 0
print('✅ 练习 4 通过: 吞吐统计能定位流水线瓶颈阶段')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def count_unique_clusters(docs, m=200, threshold=0.7, seed=0):
    a, b = make_hash_family(m, seed)
    sigs = [minhash_signature(shingles(d), a, b) for d in docs]
    keep = []
    for i in range(len(docs)):
        if not any(estimate_jaccard(sigs[i], sigs[j]) >= threshold for j in keep):
            keep.append(i)
    return len(keep)

In [ ]:
# 练习 2 参考答案
def block_mixing(n_blocks, block_size, buf_size, seed=0):
    stream = []
    for blk in range(n_blocks):
        stream.extend([blk] * block_size)
    out = shuffle_buffer(stream, buf_size, seed)
    diff = sum(out[i] != out[i + 1] for i in range(len(out) - 1))
    return diff / (len(out) - 1)

In [ ]:
# 练习 3 参考答案
def filter_corpus(docs):
    kept = [clean_text(d) for d in docs]
    kept = [d for d in kept if passes_quality(d)]
    removed = len(docs) - len(kept)
    return kept, {'kept': len(kept), 'removed': removed,
                  'removal_rate': removed / max(len(docs), 1)}

In [ ]:
# 练习 4 参考答案
import time
def throughput(items, process_fn):
    t0 = time.perf_counter()
    for x in items:
        process_fn(x)
    dt = time.perf_counter() - t0
    return len(items), len(items) / max(dt, 1e-9)

---
## 🧪 真实数据胶囊：tiny-shakespeare 上的去重与质量

用真实文本(尝试联网拉 **tiny-shakespeare**, 失败则回退到内置的真实莎翁片段)做端到端: 切成文档 → 人为注入近重复 → MinHash 去重 → 统计。真实文本上 MinHash 同样有效。

In [ ]:
def load_shakespeare():
    '''尝试联网拉 tiny-shakespeare; 失败回退到内置真实片段。'''
    try:
        import urllib.request
        url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
        with urllib.request.urlopen(url, timeout=5) as r:
            text = r.read().decode('utf-8')
        return text, 'tiny-shakespeare(联网真实)'
    except Exception:
        # 回退: 内置真实莎翁片段(Hamlet/Macbeth/R&J 真实台词)
        text = ('To be, or not to be, that is the question. '
                'Whether tis nobler in the mind to suffer the slings and arrows. '
                'Tomorrow and tomorrow and tomorrow creeps in this petty pace. '
                'But soft what light through yonder window breaks it is the east. '
                'Friends Romans countrymen lend me your ears I come to bury Caesar. '
                'All the worlds a stage and all the men and women merely players. ') * 4
        return text, '内置真实莎翁片段(回退)'

text, src = load_shakespeare()
print(f'数据来源: {src}, 共 {len(text)} 字符')

# 切成定长文档
words = text.split()
chunk = 25
base_docs = [' '.join(words[i:i + chunk]) for i in range(0, min(len(words), 25 * chunk), chunk)]
base_docs = [d for d in base_docs if len(d.split()) >= 10][:20]
# 人为注入近重复(改几个词), 模拟转载
import copy
corpus = list(base_docs)
for d in base_docs[:5]:
    ws = d.split()
    if len(ws) > 3:
        ws[2] = ws[2] + 'X'      # 改一个词 -> 近重复
    corpus.append(' '.join(ws))
print(f'注入近重复后语料: {len(corpus)} 篇 (其中 5 篇是前5篇的近重复)')

kept = dedup_near(corpus, m=256, threshold=0.6)
print(f'MinHash 去重: {len(corpus)} -> {len(kept)} 篇')
assert len(kept) <= len(base_docs) + 1, '注入的近重复应被大量去掉'
assert len(kept) < len(corpus), '去重确实减少了文档数'
print('✅ 真实文本上 MinHash 近重复去重有效: 注入的转载被识别并塌缩')

**🧪 胶囊练习**：实现 `dedup_rate(n_before, n_after)`：返回去重率 `(去掉的比例)`。

In [ ]:
def dedup_rate(n_before, n_after):
    # TODO: 返回 (n_before - n_after) / n_before
    raise NotImplementedError

In [ ]:
# 自测
rate = dedup_rate(len(corpus), len(kept))
assert 0 <= rate < 1
assert rate > 0, '确实去掉了一些近重复'
print(f'去重率 = {rate:.0%} (去掉的近重复占比)')
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def dedup_rate(n_before, n_after):
    return (n_before - n_after) / n_before

### 小结
- 预训练质量很大程度由**数据流水线**决定; 「数据多就行」是错的——清洗/过滤/去重收益常超过调架构。
- **清洗**(去HTML/控制字符/规范空白)最朴素却最值钱; **质量过滤**用长度/符号比/重复率/停用词比等弱特征组合, 没有完美过滤器。
- **去重**收益最大: 减记忆、提效率、防分布扭曲、**防评测污染**。精确去重只抓逐字相同, 真实重复多是**近重复**。
- **MinHash**: `P(签名相等) = Jaccard`, m 个哈希的签名相等率无偏估计 Jaccard, 把 O(N²) 去重降到近线性(+LSH 分桶)。
- **shuffle 缓冲**用有限内存换近似全局随机: 缓冲越大越随机, 太小打不散主题块 -> 训练有偏。
- 评测启示: 模型高分是「真会」还是「数据里背过」, 第一道防线是**去重 + 去污染**。

下一站: **模块 04 · 合成数据与模型坍塌** —— 一种特殊的「数据」: 模型自己生成的, 以及递归使用它的反噬。